# 07 — Clinical Analysis

This notebook covers group comparisons and clinical data integration (Chapter 6).

*Content to be filled with Sam's approval.*

## 1. Clinical Data Loading & Parsing

Load and parse clinical metadata for the SDS2 cohort (N=122).

See: `smartflat.utils.utils_clinical.diagnosis_logic`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from collections import Counter
from itertools import combinations
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests

from smartflat.configs.loader import import_config
from smartflat.constants import incomplete_clinical_administrations
from smartflat.constants_annotations_prototypes import mapping_cluster_category
from smartflat.datasets.utils import append_clinical_data
from smartflat.features.symbolization.main import (
    build_clusterdf,
    filter_prototypes_per_prevalence,
)
from smartflat.features.symbolization.utils import fix_clinical_diagnosis
from smartflat.features.symbolization.utils_dataset import get_experiments_dataframe
from smartflat.features.symbolization.visualization import (
    plot_cluster_prevalence,
    plot_labels_counts_per_diagnosis,
    visualize_cohort_symbolic_representation,
)
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils_visualization import plot_chronogames

In [ ]:
# --- Experiment parameters ---
annotator_id = 'samperochon'
round_number = 8

symbolization_config_name = 'SymbolicSourceInferenceCompleteGoldConfig'
config = import_config(symbolization_config_name)
clustering_config_name = config.clustering_config_name

data_root = get_data_root()
print(f"Data root: {data_root}")
print(f"Clustering config: {clustering_config_name}")

In [ ]:
df = get_experiments_dataframe(
    experiment_config_name=symbolization_config_name,
    annotator_id=annotator_id,
    round_number=round_number,
    return_symbolization=True,
    return_data_only=True,
)
print(f"Loaded {len(df)} recordings")
display(df[['identifier', 'participant_id', 'task_name', 'modality', 'N', 'duration']].head(10))

In [ ]:
df = fix_clinical_diagnosis(df)
df = df[~df['participant_id'].isin(incomplete_clinical_administrations.keys())]
df.sort_values('task_number_int', ascending=True, inplace=True)
df = df[df['pathologie'].isin(['HEALTHY', 'RIL', 'TBI'])]
df.drop_duplicates(subset=['trigram'], keep='first', inplace=True)

# Append clinical columns (age, sex, MoCA, ISDC) if missing from the pickle
clinical_cols = ['age', 'sex', 'MoCA', 'ISDC']
if not all(c in df.columns for c in clinical_cols):
    df = append_clinical_data(df, verbose=True)

print(f"After filtering: {len(df)} participants")
display(df.groupby('pathologie')['group_folder'].value_counts().to_frame('count').T)

## 2. Cohort Summary

Summarize demographics and diagnosis distribution (26 Controls, 59 TBI, 37 RIL).

See: `smartflat.utils.utils_clinical`

In [ ]:
if df is not None:
    # Encode sex as binary for descriptive statistics
    if 'sex' in df.columns:
        df['sex_encode'] = df['sex'].apply(lambda x: 1 if x == 'H' else 0 if isinstance(x, str) else np.nan)

    agg_cols = [c for c in ['age', 'sex_encode', 'MoCA', 'ISDC', 'duration'] if c in df.columns]
    demo_table = df.groupby('pathologie')[agg_cols].agg(['count', 'mean', 'std']).round(2)
    print(f"Cohort demographics (N={len(df)})")
    display(demo_table)
    display(df.groupby('pathologie').size().to_frame('n_participants'))

In [ ]:
if df is not None:
    color_palettes = {
        'sex': {'H': '#A1A1A1', 'F': '#CFCFCF'},
        'pathologie': {'HEALTHY': '#91c8e4', 'TBI': '#f7b267', 'RIL': '#f9c49a'},
    }
    pathologie_order = ['HEALTHY', 'TBI', 'RIL']

    categorical_vars = [c for c in ['sex', 'pathologie'] if c in df.columns]
    continuous_vars = [c for c in ['age', 'MoCA', 'ISDC'] if c in df.columns]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    for idx, col in enumerate(categorical_vars):
        palette = color_palettes.get(col, None)
        ax = axes[0, idx]
        sns.countplot(data=df, y=col, order=df[col].value_counts().index, ax=ax, palette=palette)
        value_counts = df[col].value_counts()
        legend_labels = [f'{cat}: {count}' for cat, count in value_counts.items()]
        ax.set_title(f'Distribution of {col}', fontsize=14, fontweight='bold')
        ax.legend(legend_labels, title="Counts")

    for idx, col in enumerate(continuous_vars):
        ax = axes[1, idx]
        sns.histplot(df[col].dropna(), bins=10, kde=True, ax=ax)
        mean, std, count = df[col].mean(), df[col].std(), df[col].count()
        ax.set_title(f'Distribution of {col}', fontsize=14, fontweight='bold')
        ax.legend([f'N={count}, mean={mean:.1f}, std={std:.1f}'])

    # Remove unused axes
    for i in range(len(categorical_vars), 3):
        fig.delaxes(axes[0, i])
    for i in range(len(continuous_vars), 3):
        fig.delaxes(axes[1, i])

    plt.suptitle(f'Cohort Demographics (N={len(df)})', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 3. Missing Data Analysis

Identify and report missing clinical fields across participants.

See: `smartflat.utils.utils_clinical.diagnosis_logic`

In [ ]:
if df is not None:
    clinical_cols = [c for c in ['age', 'sex', 'ISDC', 'MoCA', 'bras', 'pathologie'] if c in df.columns]

    missing_counts = df[clinical_cols].isna().sum()
    missing_pct = (df[clinical_cols].isna().mean() * 100).round(1)
    n_total = len(df)

    missing_df = pd.DataFrame({'Count': missing_counts, 'Percentage': missing_pct})
    display(missing_df)

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 1]})

    sns.barplot(x=missing_df.index, y=missing_df['Percentage'], palette='Set2', ax=axes[0])
    axes[0].set_title(f'Percentage of Missing Data (N={n_total})', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Percentage')
    axes[0].set_ylim(-5, 105)
    axes[0].axhline(y=100, color='red', linestyle='--', linewidth=1, label='100%')
    for i, p in enumerate(axes[0].patches):
        axes[0].text(p.get_x() + p.get_width() / 2, p.get_height() + 1,
                     f'{missing_df["Percentage"].iloc[i]:.1f}%', ha='center', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)

    sns.barplot(x=missing_df.index, y=missing_df['Count'], palette='Set2', ax=axes[1])
    axes[1].set_title(f'Count of Missing Data (N={n_total})', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

## 4. Symbolic Representation Loading

Load precomputed symbolic sequences and cluster DataFrames.

See: `smartflat.features.symbolization.utils_dataset.get_experiments_dataframe`

In [ ]:
if df is not None:
    clusterdf = build_clusterdf(
        df,
        embeddings_label_col='opt_embedding_labels',
        segments_labels_col='opt_segments_labels',
        annotator_id=annotator_id,
        round_number=round_number,
        verbose=False,
    )
    print(f"Clusters: {len(clusterdf)}")
    display(clusterdf[['cluster_index', 'cluster_type', 'n_subjects', 'n_appearance',
                        'cluster_dist_mean']].head(15))

    excluded = filter_prototypes_per_prevalence(
        clusterdf, min_subjects=10, min_occurences=10, title='Clinical analysis', verbose=True,
    )
    print(f"Excluded {len(excluded)} low-prevalence clusters")

    plot_cluster_prevalence(
        df,
        segments_labels_col='opt_segments_labels',
        title=f'Prototype prevalence (round {round_number})',
        verbose=True,
    )
    plt.show()

## 5. Per-Diagnosis Cluster Statistics

Compute cluster prevalence and occurrence statistics per diagnosis group.

See: `smartflat.features.symbolization.visualization.plot_cluster_prevalence`, `smartflat.features.symbolization.visualization.plot_labels_counts_per_diagnosis`

In [ ]:
if df is not None:
    plot_labels_counts_per_diagnosis(df, column='opt_embedding_labels')
    plt.suptitle(f'Symbol counts per diagnosis (round {round_number})', y=1.02)
    plt.show()

In [ ]:
if df is not None:
    # Build mapping: cluster_index -> cluster_type (task-definitive / exo-definitive / Noise)
    cluster_type_map = dict(zip(clusterdf['cluster_index'], clusterdf['cluster_type']))
    cluster_type_map[-1] = 'Noise'

    def _proportion_by_type(segments_labels, target_type):
        """Proportion of foreground segments classified as target_type."""
        types = [cluster_type_map.get(s, 'Unknown') for s in segments_labels]
        fg = [t for t in types if t in ('task-definitive', 'exo-definitive')]
        if len(fg) == 0:
            return np.nan
        return sum(1 for t in fg if t == target_type) / len(fg)

    df['p_exo_definitive'] = df['opt_segments_labels'].apply(
        lambda x: _proportion_by_type(x, 'exo-definitive'))
    df['p_task_definitive'] = df['opt_segments_labels'].apply(
        lambda x: _proportion_by_type(x, 'task-definitive'))

    if 'n_segments' not in df.columns:
        df['n_segments'] = df['opt_segments_labels'].apply(len)
    df['fragmentation_rate'] = df['n_segments'] / df['duration']

    print("Per-pathology descriptive statistics")
    display(
        df.groupby('pathologie')[['p_exo_definitive', 'p_task_definitive',
                                   'fragmentation_rate', 'duration']]
        .agg(['count', 'median', 'mean', 'std']).round(3)
    )

## 6. Group Comparisons (Control vs TBI vs RIL)

Statistical comparisons of symbolic features across diagnosis groups.

See: `smartflat.features.symbolization.visualization.plot_cluster_boxplots_hue`, `smartflat.utils.utils_visualization.plot_per_cat_x_cont_y_distributions`

In [ ]:
def mwu_pairwise(df, variable, group_col='pathologie', groups=None, correction='fdr_bh'):
    """
    Pairwise Mann-Whitney U tests with rank-biserial effect size and BH correction.

    Parameters
    ----------
    df : DataFrame with columns [variable, group_col]
    variable : str, column name for the continuous variable
    group_col : str, column name for the grouping variable
    groups : list, group names to compare (default: sorted unique values)
    correction : str, multiple testing correction method (default: 'fdr_bh')

    Returns
    -------
    DataFrame with U statistic, p-values (raw and corrected), rank-biserial correlation
    """
    if groups is None:
        groups = sorted(df[group_col].dropna().unique())

    results = []
    for g1, g2 in combinations(groups, 2):
        x = df.loc[df[group_col] == g1, variable].dropna()
        y = df.loc[df[group_col] == g2, variable].dropna()

        if len(x) < 2 or len(y) < 2:
            continue

        U, p = stats.mannwhitneyu(x, y, alternative='two-sided')
        n1, n2 = len(x), len(y)
        r_rb = 1 - (2 * U) / (n1 * n2)

        results.append({
            'group1': g1, 'group2': g2,
            'n1': n1, 'n2': n2,
            'median1': x.median(), 'median2': y.median(),
            'U': U, 'p_raw': p, 'r_rb': r_rb,
        })

    results_df = pd.DataFrame(results)

    if len(results_df) > 0 and correction:
        reject, p_corrected, _, _ = multipletests(results_df['p_raw'], method=correction)
        results_df['p_corrected'] = p_corrected
        results_df['significant'] = reject

    return results_df

In [ ]:
if df is not None:
    test_variables = [
        ('duration', 'Total duration (min)'),
        ('fragmentation_rate', 'Fragmentation rate (segments/min)'),
        ('p_exo_definitive', 'Proportion exo-definitive segments'),
    ]
    groups = ['HEALTHY', 'RIL', 'TBI']

    for var, label in test_variables:
        if var not in df.columns:
            continue
        print(f"\n=== {label} ===")
        result = mwu_pairwise(df, var, group_col='pathologie', groups=groups)
        display(result.round(4))

        # KDE distribution plot
        fig, ax = plt.subplots(figsize=(10, 3))
        colors = {'HEALTHY': 'tab:blue', 'RIL': 'tab:red', 'TBI': 'tab:orange'}
        for g in groups:
            sub = df.loc[df['pathologie'] == g, var].dropna()
            if len(sub) > 0:
                sns.kdeplot(sub, label=f'{g} (N={len(sub)})', color=colors[g], fill=True,
                            alpha=0.3, ax=ax)
        ax.set(xlabel=label, title=label)
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
if df is not None:
    # Explode segments into one row per segment (delta_t=8 frames, fps=25)
    df['raw_segments_length'] = df['raw_cpts'].apply(np.ediff1d) * 8 / 25

    long_df = df[['participant_id', 'pathologie', 'raw_segments_length']].explode('raw_segments_length')
    long_df['raw_segments_length'] = long_df['raw_segments_length'].astype(float)
    long_df = long_df[long_df['raw_segments_length'] > 0]
    long_df['log_len'] = np.log(long_df['raw_segments_length'])

    print(f"Total segments: {len(long_df)}")
    display(long_df.groupby('pathologie').agg(
        n_participants=('participant_id', 'nunique'),
        n_segments=('log_len', 'count'),
        median_s=('raw_segments_length', 'median'),
        mean_s=('raw_segments_length', 'mean'),
    ).round(2))

    # --- ECDF per pathology group ---
    fig, ax = plt.subplots(figsize=(14, 5))

    for pid in df['participant_id'].unique():
        segs = long_df.loc[long_df['participant_id'] == pid, 'raw_segments_length']
        sns.ecdfplot(segs, color='gray', alpha=0.3, linewidth=0.5, ax=ax)

    colors = {'HEALTHY': 'tab:blue', 'RIL': 'tab:red', 'TBI': 'tab:orange'}
    for patho, color in colors.items():
        sub = long_df[long_df['pathologie'] == patho]
        if len(sub) > 0:
            sns.ecdfplot(sub['raw_segments_length'], label=patho, color=color,
                         linewidth=2.5, ax=ax)

    ax.set_xscale('log')
    ax.set(xlabel='Segment duration (seconds)', ylabel='ECDF',
           title='Empirical CDF of segment durations per pathology group')
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

    # --- Linear mixed-effects model ---
    null = smf.mixedlm('log_len ~ 1', long_df, groups=long_df['participant_id'])
    res_null = null.fit(reml=False)

    alt = smf.mixedlm('log_len ~ C(pathologie)', long_df, groups=long_df['participant_id'])
    res_alt = alt.fit(reml=False)

    llr = 2 * (res_alt.llf - res_null.llf)
    df_diff = res_alt.df_modelwc - res_null.df_modelwc
    pval = stats.chi2.sf(llr, df_diff)

    print(f"\nLikelihood ratio test: LR={llr:.3f}, df={df_diff}, p={pval:.4e}")
    print(f"\nFixed effects (log scale):")
    for param, coef in res_alt.fe_params.items():
        print(f"  {param}: coef={coef:.4f}, multiplicative effect={np.exp(coef)-1:.1%}")

    var_u = res_alt.cov_re.iloc[0, 0]
    var_eps = res_alt.scale
    icc = var_u / (var_u + var_eps)
    print(f"\nICC (participant): {icc:.3f}")
    print(res_alt.summary())

## 7. Visualization

Plot chronograms by diagnosis, boxplots, and distribution comparisons.

See: `smartflat.features.symbolization.visualization.visualize_cohort_symbolic_representation`

In [ ]:
if df is not None:
    visualize_cohort_symbolic_representation(
        df,
        covariate='pathologie',
        labels_col='opt_embedding_labels',
    )
    plt.suptitle(f'Cohort symbolic representation by pathology (round {round_number})', y=1.02)
    plt.show()

In [ ]:
if df is not None:
    for patho in ['HEALTHY', 'TBI', 'RIL']:
        df_sub = df[df['pathologie'] == patho].copy()
        if len(df_sub) > 0:
            plot_chronogames(df_sub, labels_col='opt_embedding_labels',
                             n_subjects=min(8, len(df_sub)))
            plt.suptitle(f'{patho} — Symbolic sequences (N={len(df_sub)})', y=1.02)
            plt.show()